# Binary search: three boundaries, and an answer space

Everyone can write "check the middle, throw away half". Almost nobody can write it
correctly the first time, because the loop has three decisions and only one of them
is the interesting one:

| decision | choices | consequence of getting it wrong |
|---|---|---|
| the interval | `[lo, hi]` closed vs `[lo, hi)` half-open | infinite loop, or an off-by-one at the ends |
| the update | `lo = mid` / `lo = mid + 1`, `hi = mid` / `hi = mid - 1` | infinite loop, or a skipped answer |
| the comparison | `<` vs `<=` | the *other* boundary - silently wrong at every tie |

This notebook uses one interval convention for everything: **half-open `[lo, hi)`**,
with the invariant

    a[:lo]  is known too small
    a[hi:]  is known big enough
    the answer lives in [lo, hi)

When the window empties, `lo == hi` *is* the answer. There is no early exit, so there
is no separate found/not-found case to get wrong.

In [1]:
from bisect import bisect_left, bisect_right

def lower_bound(a, x):
    # first index i with a[i] >= x   (== len(a) if x is bigger than everything)
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1      # a[mid] is too small -> discard it too
        else:
            hi = mid          # a[mid] might BE the answer -> keep it
    return lo

def upper_bound(a, x):
    # first index i with a[i] > x.  One character different: <= instead of <
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def find(a, x):
    i = lower_bound(a, x)
    return i if i < len(a) and a[i] == x else -1

def count_of(a, x):
    return upper_bound(a, x) - lower_bound(a, x)

def last_le(a, x):
    # largest index with a[i] <= x, or -1
    return upper_bound(a, x) - 1

a = [2, 3, 3, 3, 5, 8, 8, 13]
print('a =', a)
print('  x | lower upper count  find last<=')
for x in [1, 2, 3, 4, 8, 13, 20]:
    lb, ub = lower_bound(a, x), upper_bound(a, x)
    assert (lb, ub) == (bisect_left(a, x), bisect_right(a, x))
    print(f'{x:>3} | {lb:>5} {ub:>5} {count_of(a, x):>5} {find(a, x):>5} {last_le(a, x):>6}')

a = [2, 3, 3, 3, 5, 8, 8, 13]
  x | lower upper count  find last<=
  1 |     0     0     0    -1     -1
  2 |     0     1     1     0      0
  3 |     1     4     3     1      3
  4 |     4     4     0    -1      3
  8 |     5     7     2     5      6
 13 |     7     8     1     7      7
 20 |     8     8     0    -1      7


`lower_bound` is Python's `bisect_left`, `upper_bound` is `bisect_right`. They differ
only for values that actually exist; for a missing value they collapse to the same
insertion point. Everything else is a one-liner on top:

- exists? `a[lower_bound(x)] == x`
- how many? `upper_bound(x) - lower_bound(x)`
- **which bucket does x fall into?** `upper_bound(x) - 1`

That last one is the one that shows up in real systems code, and the `- 1` is not
cosmetic.

## Three loops that look right

Two of them hang. The third returns a plausible number.

In [2]:
STEP_CAP = 100

def broken_never_shrinks(a, x):
    # closed interval [lo, hi] but assigns hi = mid -> hi never moves past mid
    lo, hi, steps = 0, len(a) - 1, 0
    while lo <= hi:
        steps += 1
        if steps > STEP_CAP:
            return None, steps
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo, steps

def broken_lo_equals_mid(a, x):
    # half-open but assigns lo = mid; when hi - lo == 1, mid == lo and nothing moves
    lo, hi, steps = 0, len(a), 0
    while lo < hi:
        steps += 1
        if steps > STEP_CAP:
            return None, steps
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid
        else:
            hi = mid
    return lo, steps

def broken_skips_answer(a, x):
    # terminates, and quietly throws away the element it was supposed to keep
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return lo

a2 = [1, 2, 3, 4, 5, 6, 7]
print('a =', a2, ' target 4, correct lower_bound =', lower_bound(a2, 4))
print('(a) while lo <= hi with hi = mid     ->', broken_never_shrinks(a2, 4))
print('(b) lo = mid instead of lo = mid + 1 ->', broken_lo_equals_mid(a2, 4))
print('(c) hi = mid - 1                     ->', broken_skips_answer(a2, 4), '<- wrong, and it terminates')

a = [1, 2, 3, 4, 5, 6, 7]  target 4, correct lower_bound = 3
(a) while lo <= hi with hi = mid     -> (None, 101)
(b) lo = mid instead of lo = mid + 1 -> (None, 101)
(c) hi = mid - 1                     -> 2 <- wrong, and it terminates


And one that cannot happen in Python but happens in every language with fixed-width
integers - including the CUDA kernels these servers are made of.

In [3]:
def add_int32(x, y):
    s = (x + y) & 0xFFFFFFFF
    return s - (1 << 32) if s >= (1 << 31) else s

lo, hi = 2_000_000_000, 2_100_000_000
print('lo + hi in int32      =', add_int32(lo, hi))
print('(lo + hi) // 2        =', add_int32(lo, hi) // 2, '<- negative index')
print('lo + (hi - lo) // 2   =', lo + (hi - lo) // 2)

lo + hi in int32      = -194967296
(lo + hi) // 2        = -97483648 <- negative index
lo + (hi - lo) // 2   = 2050000000


## Binary search on the answer

The array is a special case. The only thing the loop actually needs is a **monotone
predicate**: a function that is `False, False, ..., False, True, True, ..., True` over
the range. A sorted array just happens to give you one for free (`a[i] >= x`).

So the recipe becomes:

1. decide what the answer is a number *of* - hours, bananas per hour, bytes, qps
2. write `ok(candidate)` - usually a simulation, not a comparison
3. **argue that `ok` is monotone**
4. run the same loop over the candidate range

Step 3 is the whole job. If `ok` is not monotone the loop still terminates and still
returns a number, and that number means nothing.

In [4]:
def search_first_true(lo, hi, ok):
    # smallest x in [lo, hi) with ok(x) true - the same loop as lower_bound
    probes = []
    while lo < hi:
        mid = lo + (hi - lo) // 2
        good = ok(mid)
        probes.append((mid, good))
        if good:
            hi = mid
        else:
            lo = mid + 1
    return lo, probes

def koko_hours(piles, speed):
    return sum(-(-p // speed) for p in piles)      # ceil division

piles, h = [30, 11, 23, 4, 20], 6
ans, probes = search_first_true(1, max(piles) + 1, lambda s: koko_hours(piles, s) <= h)
print('LC 875 answer:', ans, 'bananas/hour in', len(probes), 'probes')
print()
print('the array we never built:')
print('speed ' + ''.join(f'{s:>4}' for s in range(1, 31, 2)))
print('hours ' + ''.join(f'{koko_hours(piles, s):>4}' for s in range(1, 31, 2)))
print('ok    ' + ''.join(f"{'T' if koko_hours(piles, s) <= h else '.':>4}" for s in range(1, 31, 2)))
print()
for mid, good in probes:
    print(f'  speed={mid:>3} -> {koko_hours(piles, mid):>3} hours ->',
          'fast enough, try slower' if good else 'too slow, speed up')

LC 875 answer: 23 bananas/hour in 5 probes

the array we never built:
speed    1   3   5   7   9  11  13  15  17  19  21  23  25  27  29
hours   88  31  19  15  13  10   9   8   8   8   7   6   6   6   6
ok       .   .   .   .   .   .   .   .   .   .   .   T   T   T   T

  speed= 16 ->   8 hours -> too slow, speed up
  speed= 24 ->   6 hours -> fast enough, try slower
  speed= 20 ->   7 hours -> too slow, speed up
  speed= 22 ->   7 hours -> too slow, speed up
  speed= 23 ->   6 hours -> fast enough, try slower


In [5]:
def can_split(nums, m, cap):
    # LC 410: can nums be cut into <= m contiguous chunks each summing <= cap?
    chunks, cur = 1, 0
    for v in nums:
        if cur + v > cap:
            chunks += 1
            cur = 0
        cur += v
    return chunks <= m

nums, m = [7, 2, 5, 10, 8], 2
cap, probes410 = search_first_true(max(nums), sum(nums) + 1, lambda c: can_split(nums, m, c))
print('LC 410 answer:', cap, 'in', len(probes410), 'probes')
print('range floor  :', max(nums), '(one chunk must hold the largest element)')
print('range ceiling:', sum(nums), '(one chunk holds everything)')

LC 410 answer: 18 in 4 probes
range floor  : 10 (one chunk must hold the largest element)
range ceiling: 32 (one chunk holds everything)


## When one probe costs three minutes

SGLang's autotuner (`python/sglang/auto_benchmark_lib.py`) binary searches the request
rate: *does the server still meet its SLA at this qps?* Every answer is a full
benchmark run, so halving the range is worth real wall-clock time - and two things
change once the predicate is a measurement rather than a comparison.

    while upper - lower > tolerance and rounds_run < max_rounds:
        qps = pick_qps_midpoint(lower, upper)
        if qps <= lower or qps >= upper:
            break
        record = one_trial(qps, max_concurrency)
        if record["sla_passed"]:
            lower = qps
            best = record
        else:
            upper = qps

First, a float interval can always be halved again, so the loop terminates on a
**tolerance**, not on `lo < hi`. Second, the midpoint is rounded, so it can land on an
endpoint and stall - hence the explicit `break`. That guard is the float twin of
`lo = mid` hanging on integers.

In [6]:
import random

def bisect_float(lo, hi, ok, tolerance, max_rounds=64):
    rounds, best = 0, None
    while hi - lo > tolerance and rounds < max_rounds:
        mid = round((lo + hi) / 2, 4)
        if mid <= lo or mid >= hi:       # midpoint collapsed onto an endpoint
            break
        if ok(mid):
            lo, best = mid, mid
        else:
            hi = mid
        rounds += 1
    return best, lo, hi, rounds

def make_sla_probe(true_capacity, jitter=0.0, seed=0):
    rng, calls = random.Random(seed), []
    def ok(qps):
        measured = true_capacity + (rng.uniform(-jitter, jitter) if jitter else 0.0)
        calls.append((qps, round(measured, 3), qps <= measured))
        return qps <= measured
    return ok, calls

cap_true = 13.7
ok, calls = make_sla_probe(cap_true)
best, lo_, hi_, rounds = bisect_float(1.0, 64.0, ok, tolerance=0.1)
print(f'clean predicate  : {rounds} runs -> {best} qps   bracket [{lo_}, {hi_}]')
print('  probes:', [c[0] for c in calls])

ok_n, calls_n = make_sla_probe(cap_true, jitter=0.6, seed=23)
best_n, lo_n, hi_n, rounds_n = bisect_float(1.0, 64.0, ok_n, tolerance=0.1)
print(f'noisy predicate  : {rounds_n} runs -> {best_n} qps   bracket [{lo_n}, {hi_n}]')
for qps, measured, verdict in calls_n:
    if (qps <= cap_true) != verdict:
        print(f'  disagreed: qps={qps} measured capacity {measured} -> '
              f"{'PASS' if verdict else 'FAIL'}")

clean predicate  : 10 runs -> 13.6738 qps   bracket [13.6738, 13.7353]
  probes: [32.5, 16.75, 8.875, 12.8125, 14.7812, 13.7968, 13.3046, 13.5507, 13.6738, 13.7353]
noisy predicate  : 10 runs -> 13.3661 qps   bracket [13.3661, 13.4276]
  disagreed: qps=13.5507 measured capacity 13.256 -> FAIL
  disagreed: qps=13.4276 measured capacity 13.33 -> FAIL


A wrong answer near the middle of the search is unrecoverable - binary search never
revisits a discarded half. That is why the real loop carries a round budget and keeps
the passing *record*, not just the number.

## Galloping: when you expect the answer near the front

SGLang's radix cache answers "how long a prefix do these two token sequences share?"
on every incoming request (`srt/mem_cache/radix_cache.py`, `RadixKey.match`). A
per-token Python loop is out of the question, so it does an **exponential search**:
compare slices of length 1, 2, 4, 8, ... until one differs, then binary search inside
that one window.

In [7]:
def match_linear(t0, t1):
    n = min(len(t0), len(t1))
    i = 0
    while i < n and t0[i] == t1[i]:
        i += 1
    return i, i + (0 if i == n else 1), i + (0 if i == n else 1)

def match_binary(t0, t1):
    n = min(len(t0), len(t1))
    probes = tokens = 0
    lo, hi = 0, n
    while lo < hi:
        mid = lo + (hi - lo) // 2
        probes += 1
        tokens += mid + 1                      # a probe compares a PREFIX, not an element
        if t0[:mid + 1] == t1[:mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo, probes, tokens

def match_gallop(t0, t1):
    n = min(len(t0), len(t1))
    probes = tokens = 0
    matched, lo, step = n, 0, 1
    while lo < n:
        hi = lo + step if lo + step < n else n
        probes += 1
        tokens += hi - lo
        if t0[lo:hi] != t1[lo:hi]:
            # divergence is inside [lo, hi).  the condition is hi - lo > 1, so mid > lo
            # strictly and `lo = mid` is safe here - the window still shrinks
            while hi - lo > 1:
                mid = (lo + hi) // 2
                probes += 1
                tokens += mid - lo
                if t0[lo:mid] == t1[lo:mid]:
                    lo = mid
                else:
                    hi = mid
            matched = lo
            break
        lo = hi
        step *= 2
    return matched, probes, tokens

n = 4096
base = list(range(n))
print(f'{"p":>6} | {"linear":>17} | {"binary":>17} | {"gallop":>17}')
print(f'{"":>6} | {"probes":>8}{"tokens":>9} | {"probes":>8}{"tokens":>9} | {"probes":>8}{"tokens":>9}')
for p in [0, 3, 17, 250, 2048, 4095, 4096]:
    other = base[:p] + [-1] * (n - p)
    cells = []
    for fn in (match_linear, match_binary, match_gallop):
        got, probes, tokens = fn(base, other)
        assert got == min(p, n)
        cells.append(f'{probes:>8}{tokens:>9}')
    print(f'{p:>6} | ' + ' | '.join(cells))

     p |            linear |            binary |            gallop
       |   probes   tokens |   probes   tokens |   probes   tokens
     0 |        1        1 |       13     4108 |        1        1
     3 |        4        4 |       12     4109 |        5       10
    17 |       18       18 |       12     4171 |        9       46
   250 |      251      251 |       12     5635 |       15      382
  2048 |     2049     2049 |       12    22541 |       23     6142
  4095 |     4096     4096 |       12    45069 |       13     4096
  4096 |     4096     4096 |       12    45069 |       13     4096


Two cost models, two different winners. Counting **probes**, plain binary search looks
unbeatable - 12 or 13 whatever `p` is. But a probe compares a prefix, so it costs
`O(mid)`: probes are neither free nor equal. Counting **tokens touched**, binary search
does about `n log n` work while galloping does `O(p)`, because the doubling windows sum
to roughly `2p` and the binary phase only searches the last one.

Galloping is not uniformly better - around `p ~ n/2` it makes about twice as many
probes. It is the right bet for a prefix cache because that workload is bimodal: a new
conversation shares almost nothing, a follow-up turn shares almost everything, and
hardly anything lands in the middle.

## The inverse of a prefix sum is a binary search

Counting sort turned a histogram into an offset table with an exclusive scan. Every
batched kernel in an inference server carries that table as `cu_seqlens`, and every
kernel that works one token at a time has to undo it: *given a flat token index, which
request does it belong to?*

    seq_of = torch.searchsorted(cu_seqlens, tok, right=True) - 1

`right=True` is not a style choice.

In [8]:
def seq_of_tokens(cu, n_tokens, right=True):
    ub = bisect_right if right else bisect_left
    return [ub(cu, tok) - 1 for tok in range(n_tokens)]

seq_lens = [3, 5, 2]
cu = [0]
for L in seq_lens:
    cu.append(cu[-1] + L)
total = cu[-1]
truth = [i for i, L in enumerate(seq_lens) for _ in range(L)]

print('seq_lens   =', seq_lens)
print('cu_seqlens =', cu)
print('token       ' + ''.join(f'{t:>4}' for t in range(total)))
print('truth       ' + ''.join(f'{v:>4}' for v in truth))
print('right=True  ' + ''.join(f'{v:>4}' for v in seq_of_tokens(cu, total, True)))
print('right=False ' + ''.join(f'{v:>4}' for v in seq_of_tokens(cu, total, False)))
wrong = [t for t in range(total) if seq_of_tokens(cu, total, False)[t] != truth[t]]
print('wrong with right=False:', wrong, '- the first token of every request')

seq_lens   = [3, 5, 2]
cu_seqlens = [0, 3, 8, 10]
token          0   1   2   3   4   5   6   7   8   9
truth          0   0   0   1   1   1   1   1   2   2
right=True     0   0   0   1   1   1   1   1   2   2
right=False   -1   0   0   0   1   1   1   1   1   2
wrong with right=False: [0, 3, 8] - the first token of every request


`bisect_right(cu, tok) - 1` is *the last start that is `<= tok`*, which is the
definition of "which bucket am I in". `bisect_left` asks for the first start `>= tok`,
and at a boundary that is the **next** request. Token 0 gets id `-1`, which then
indexes the end of the table and reads another request's state - no crash, wrong
numbers.

## Rounding up to a bucket

A CUDA graph is captured for a fixed batch size, so a decode step with 37 requests has
to replay on the smallest captured shape that can hold it
(`srt/model_executor/runner/base_cuda_graph_runner.py`):

    index = bisect.bisect_left(buckets, raw_size)
    return buckets[index]

`bisect_left`, not `bisect_right`. If `raw_size` is already a captured size,
`bisect_left` returns it and nothing is padded; `bisect_right` steps to the next bucket
and pays for a whole graph of empty rows.

In [9]:
# the default decode capture list, from server_args._generate_decode_cuda_graph_batch_sizes
NORMAL_BS = [1, 2, 4, 8, 12] + list(range(16, 257, 8))
SPEC_BS = (list(range(1, 9)) + list(range(10, 33, 2))
           + list(range(40, 65, 4)) + list(range(72, 257, 8)))

def pad_to_bucket(raw_size, buckets, right=False):
    assert raw_size <= buckets[-1]
    return buckets[(bisect_right if right else bisect_left)(buckets, raw_size)]

def padding_waste(buckets, hi=255):
    return sum((pad_to_bucket(b, buckets) - b) / pad_to_bucket(b, buckets)
               for b in range(1, hi + 1)) / hi

print(f'{"batch":>6} | {"bisect_left":>11} {"waste":>7} | {"bisect_right":>12} {"waste":>7}')
for bs in [1, 3, 12, 16, 17, 24, 100, 249]:
    L, R = pad_to_bucket(bs, NORMAL_BS), pad_to_bucket(bs, NORMAL_BS, right=True)
    mark = '   <- exactly a captured size' if bs in NORMAL_BS else ''
    print(f'{bs:>6} | {L:>11} {(L - bs) / L:>6.1%} | {R:>12} {(R - bs) / R:>6.1%}{mark}')

wr = sum((pad_to_bucket(b, NORMAL_BS, True) - b) / pad_to_bucket(b, NORMAL_BS, True)
         for b in range(1, 256)) / 255
print()
print(f'averaged over batch 1..255:  bisect_left {padding_waste(NORMAL_BS):.2%}'
      f'   bisect_right {wr:.2%}')
print(f'spec-decode bucket list, bisect_left: {padding_waste(SPEC_BS):.2%}')
try:
    pad_to_bucket(256, NORMAL_BS, right=True)
except (IndexError, AssertionError) as e:
    print('and at the largest batch, bisect_right runs off the end:', type(e).__name__)

 batch | bisect_left   waste | bisect_right   waste
     1 |           1   0.0% |            2  50.0%   <- exactly a captured size
     3 |           4  25.0% |            4  25.0%
    12 |          12   0.0% |           16  25.0%   <- exactly a captured size
    16 |          16   0.0% |           24  33.3%   <- exactly a captured size
    17 |          24  29.2% |           24  29.2%
    24 |          24   0.0% |           32  25.0%   <- exactly a captured size
   100 |         104   3.8% |          104   3.8%
   249 |         256   2.7% |          256   2.7%

averaged over batch 1..255:  bisect_left 4.25%   bisect_right 6.07%
spec-decode bucket list, bisect_left: 2.63%
and at the largest batch, bisect_right runs off the end: IndexError


The spec-decoding list is denser below 64 for a reason: with a draft model every batch
is multiplied by the number of speculated tokens, so a fat bucket at small batch sizes
is where the waste actually lands. Same lesson as block padding - you cannot pick
bucket boundaries without knowing the distribution of what you are bucketing.

## The interview versions

**LC 34** is why having two boundaries pays: with `lower_bound` and `upper_bound` it is
four tokens of logic, and "not found" falls out of `lo == hi` instead of needing its
own branch.

**LC 33** is the reminder that *sorted* is stronger than binary search needs. A rotated
array is not monotone, but at every `mid` one of the two halves is, and "is the target
in that half" is decidable - so each step still discards half the array.

In [10]:
def search_range(nums, target):
    lo, hi = lower_bound(nums, target), upper_bound(nums, target)
    return [lo, hi - 1] if lo < hi else [-1, -1]

def search_rotated(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:                 # left half is sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                     # right half is sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

nums34 = [5, 7, 7, 8, 8, 8, 10]
for t in [8, 7, 6, 5, 10]:
    print(f'search_range({nums34}, {t}) = {search_range(nums34, t)}')
print()
rot = [4, 5, 6, 7, 0, 1, 2]
for t in [0, 4, 2, 3, 7]:
    print(f'search_rotated({rot}, {t}) = {search_rotated(rot, t):>2}')

search_range([5, 7, 7, 8, 8, 8, 10], 8) = [3, 5]
search_range([5, 7, 7, 8, 8, 8, 10], 7) = [1, 2]
search_range([5, 7, 7, 8, 8, 8, 10], 6) = [-1, -1]
search_range([5, 7, 7, 8, 8, 8, 10], 5) = [0, 0]
search_range([5, 7, 7, 8, 8, 8, 10], 10) = [6, 6]

search_rotated([4, 5, 6, 7, 0, 1, 2], 0) =  4
search_rotated([4, 5, 6, 7, 0, 1, 2], 4) =  0
search_rotated([4, 5, 6, 7, 0, 1, 2], 2) =  6
search_rotated([4, 5, 6, 7, 0, 1, 2], 3) = -1
search_rotated([4, 5, 6, 7, 0, 1, 2], 7) =  3


## Complexity

| | time | note |
|---|---|---|
| `lower_bound` / `upper_bound` | O(log n) | ~20 probes at a million elements |
| binary search on an answer range `R` | O(log R) x cost of `ok` | the predicate dominates |
| float bisection to tolerance `t` | O(log((hi-lo)/t)) | plus a round budget when `ok` is noisy |
| galloping to a shared prefix `p` | O(log p) probes, O(p) elements | vs O(log n) probes, O(n log n) elements |
| sorting first, then searching once | O(n log n) | a single search never pays for the sort |

## Tests

In [11]:
assert lower_bound([2,3,3,3,5,8,8,13], 3) == 1 and upper_bound([2,3,3,3,5,8,8,13], 3) == 4
assert broken_never_shrinks([1,2,3,4,5,6,7], 4)[0] is None
assert broken_lo_equals_mid([1,2,3,4,5,6,7], 4)[0] is None
assert broken_skips_answer([1,2,3,4,5,6,7], 4) == 2 != lower_bound([1,2,3,4,5,6,7], 4)
assert add_int32(2_000_000_000, 2_100_000_000) < 0
assert ans == 23 and koko_hours(piles, 23) <= h < koko_hours(piles, 22)
assert cap == 18 and can_split(nums, m, 18) and not can_split(nums, m, 17)
assert best is not None and abs(best - cap_true) < 0.4
assert match_gallop(base, base[:250] + [-1] * (n - 250))[0] == 250
assert seq_of_tokens(cu, total, True) == truth and seq_of_tokens(cu, total, False) != truth
assert pad_to_bucket(17, NORMAL_BS) == 24 and pad_to_bucket(16, NORMAL_BS) == 16
assert pad_to_bucket(16, NORMAL_BS, right=True) == 24
assert search_range([5,7,7,8,8,8,10], 8) == [3, 5] and search_range([5,7,7,8,8,8,10], 6) == [-1, -1]
assert search_rotated([4,5,6,7,0,1,2], 0) == 4 and search_rotated([4,5,6,7,0,1,2], 3) == -1
print('all assertions passed')

all assertions passed
